# RQ3 Main Model Selection

This notebook estimates **candidate main models only** for RQ3. It stops before interaction modelling. The purpose is to compare alternative country fixed-effect specifications and decide which model should become the main model before running regional heterogeneity, interactions, and robustness extensions.

All candidate models use the same complete-case sample, country fixed effects, and country-clustered standard errors. Climate controls are restricted to `total_HDD` and `total_CDD`; development/scale controls are `log_GDP_pc` and `log_population`.

In [ ]:

from pathlib import Path
import json
import sys
import warnings

PROJECT = Path('/Users/nico/Research/Energy sufficiency')
LOCAL_PKG = PROJECT / 'Process/python_packages'
if LOCAL_PKG.exists() and str(LOCAL_PKG) not in sys.path:
    sys.path.insert(0, str(LOCAL_PKG))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_theme(style='whitegrid', font='DejaVu Sans')

DATA_PATH = PROJECT / 'Data/global_SE_EC_sufficiency_controls_2015.csv'
RESULTS_DIR = PROJECT / 'Results/RQ3_social_determinants/Main_model_selection'
TABLE_DIR = RESULTS_DIR / 'Tables'
FIG_DIR = RESULTS_DIR / 'Figures'
PROCESS_DIR = PROJECT / 'Process/RQ3_main_model_selection'
for d in [RESULTS_DIR, TABLE_DIR, FIG_DIR, PROCESS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('Data:', DATA_PATH)
print('Results:', RESULTS_DIR)


In [ ]:

def clean_model_data(df):
    d = df.copy()
    d['income_clean'] = d['income'].where(d['income'] >= 0, np.nan)
    d['wealth_clean'] = d['wealth'].where(d['wealth'] >= 0, np.nan)
    d['SES_raw'] = d['income_clean'].combine_first(d['wealth_clean'])
    d['female_share'] = d['gender']
    d['log_sufficiency_ratio'] = np.log(d['sufficiency_ratio'].where(d['sufficiency_ratio'] > 0))
    d['GDP_pc'] = d['GDP'] / d['population'].replace(0, np.nan)
    d['log_GDP_pc'] = np.log(d['GDP_pc'].where(d['GDP_pc'] > 0))
    d['log_population'] = np.log(d['population'].where(d['population'] > 0))
    return d

df_raw = pd.read_csv(DATA_PATH)
df = clean_model_data(df_raw)

controls = ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD']
core_predictors = ['SES_raw', 'rural', 'education', 'female_share']
optional_demographics = ['age', 'Hhsize']
full_predictors = core_predictors + optional_demographics + controls

candidate_specs = {
    'M1_full_age_hhsize': core_predictors + ['age', 'Hhsize'] + controls,
    'M2_drop_age': core_predictors + ['Hhsize'] + controls,
    'M3_drop_hhsize': core_predictors + ['age'] + controls,
    'M4_drop_age_hhsize': core_predictors + controls,
}

# Common sample across all candidate specs for fair comparison.
common_required = ['log_sufficiency_ratio', 'iso_code', 'country', 'GDLcode', 'analytical_region', 'population'] + full_predictors
sample = df.dropna(subset=common_required).copy()
sample = sample[np.isfinite(sample['log_sufficiency_ratio']) & (sample['population'] > 0)].copy()
print('Common model sample rows:', len(sample))
print('Countries:', sample['iso_code'].nunique())
print('Regions:', sample['analytical_region'].nunique())
print(sample.groupby('analytical_region').size())

sample[['GDLcode','iso_code','country','analytical_region','population','log_sufficiency_ratio'] + full_predictors].to_csv(PROCESS_DIR / 'rq3_main_model_selection_common_sample.csv', index=False)


In [ ]:

# Standardise continuous predictors on the common sample. Coefficients are therefore per 1 SD increase.
standardized = sample.copy()
standardization_rows = []
for var in sorted(set(full_predictors)):
    mu = standardized[var].mean()
    sd = standardized[var].std(ddof=0)
    standardized['z_' + var] = (standardized[var] - mu) / sd
    standardization_rows.append({'variable': var, 'mean': float(mu), 'sd': float(sd)})
standardization = pd.DataFrame(standardization_rows)
standardization.to_csv(PROCESS_DIR / 'rq3_main_model_standardization_parameters.csv', index=False)
standardization


In [ ]:

def star(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''

def pct_change(beta):
    return (np.exp(beta) - 1) * 100

def compute_vif(data, variables):
    x = data[['z_' + v for v in variables]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    X = sm.add_constant(x, has_constant='add')
    rows = []
    for idx, col in enumerate(X.columns):
        if col == 'const':
            continue
        vif = variance_inflation_factor(X.values, idx)
        rows.append({'variable': col.replace('z_', ''), 'VIF': float(vif), 'n_complete': int(len(x))})
    return pd.DataFrame(rows)

models = {}
coef_rows = []
summary_rows = []
vif_rows = []
resid_frames = []

for model_name, vars_ in candidate_specs.items():
    rhs = ' + '.join(['z_' + v for v in vars_]) + ' + C(iso_code)'
    formula = 'log_sufficiency_ratio ~ ' + rhs
    fit = smf.ols(formula, data=standardized).fit(
        cov_type='cluster',
        cov_kwds={'groups': standardized['iso_code']}
    )
    models[model_name] = fit
    resid = standardized[['GDLcode','iso_code','country','analytical_region']].copy()
    resid['model'] = model_name
    resid['residual'] = fit.resid
    resid_frames.append(resid)

    vif = compute_vif(standardized, vars_)
    vif.insert(0, 'model', model_name)
    vif_rows.append(vif)

    summary_rows.append({
        'model': model_name,
        'predictors': ', '.join(vars_),
        'n_obs': int(fit.nobs),
        'n_countries': int(standardized['iso_code'].nunique()),
        'n_regions': int(standardized['analytical_region'].nunique()),
        'r_squared': float(fit.rsquared),
        'adj_r_squared': float(fit.rsquared_adj),
        'aic': float(fit.aic),
        'bic': float(fit.bic),
        'df_model': float(fit.df_model),
        'df_resid': float(fit.df_resid),
        'max_vif': float(vif['VIF'].max()),
        'max_vif_variable': vif.sort_values('VIF', ascending=False)['variable'].iloc[0],
    })

    for var in vars_:
        param = 'z_' + var
        beta = fit.params.get(param, np.nan)
        se = fit.bse.get(param, np.nan)
        p = fit.pvalues.get(param, np.nan)
        ci_low, ci_high = fit.conf_int().loc[param].tolist()
        coef_rows.append({
            'model': model_name,
            'variable': var,
            'coef_log_points_per_sd': float(beta),
            'cluster_se': float(se),
            'ci_low': float(ci_low),
            'ci_high': float(ci_high),
            'p_value': float(p),
            'stars': star(p),
            'pct_change_per_sd': float(pct_change(beta)),
            'pct_ci_low': float(pct_change(ci_low)),
            'pct_ci_high': float(pct_change(ci_high)),
        })

coef_table = pd.DataFrame(coef_rows)
summary_table = pd.DataFrame(summary_rows)
vif_table = pd.concat(vif_rows, ignore_index=True)
residuals = pd.concat(resid_frames, ignore_index=True)

coef_table.to_csv(TABLE_DIR / 'rq3_main_model_candidate_coefficients_clustered.csv', index=False)
summary_table.to_csv(TABLE_DIR / 'rq3_main_model_candidate_summary.csv', index=False)
vif_table.to_csv(TABLE_DIR / 'rq3_main_model_candidate_vif.csv', index=False)
residuals.to_csv(PROCESS_DIR / 'rq3_main_model_candidate_residuals.csv', index=False)
summary_table


In [ ]:

# Spatial residual Moran's I for each candidate model.
spatial_rows = []
try:
    import geopandas as gpd
    from libpysal.weights import Queen, KNN
    from esda.moran import Moran

    shp_path = Path('/Users/nico/Documents/PhD research/energy poverty/Data/GDL Shapefiles V4/GDL Shapefiles V4.shp')
    gdf = gpd.read_file(shp_path, columns=['GDLcode', 'geometry'])
    spatial_base = gdf.merge(standardized[['GDLcode']], on='GDLcode', how='inner').reset_index(drop=True)
    wq = Queen.from_dataframe(spatial_base, use_index=False)
    wq.transform = 'r'
    wk = KNN.from_dataframe(spatial_base, k=6)
    wk.transform = 'r'

    for model_name in candidate_specs:
        y = residuals[residuals['model'] == model_name].set_index('GDLcode').loc[spatial_base['GDLcode'], 'residual'].values
        moran_q = Moran(y, wq, permutations=999, two_tailed=True)
        moran_k = Moran(y, wk, permutations=999, two_tailed=True)
        spatial_rows.extend([
            {'model': model_name, 'weights': 'Queen contiguity', 'n_obs': len(y), 'islands': len(wq.islands), 'moran_I': moran_q.I, 'expected_I': moran_q.EI, 'p_norm': moran_q.p_norm, 'p_sim': moran_q.p_sim},
            {'model': model_name, 'weights': 'KNN k=6', 'n_obs': len(y), 'islands': 0, 'moran_I': moran_k.I, 'expected_I': moran_k.EI, 'p_norm': moran_k.p_norm, 'p_sim': moran_k.p_sim},
        ])
except Exception as e:
    spatial_rows.append({'model': 'ALL', 'weights': 'ERROR', 'error': repr(e)})

spatial_table = pd.DataFrame(spatial_rows)
spatial_table.to_csv(TABLE_DIR / 'rq3_main_model_candidate_spatial_morans_I.csv', index=False)
spatial_table


In [ ]:

# Stability table for core variables across candidate models.
core_for_stability = core_predictors + controls
stability = coef_table[coef_table['variable'].isin(core_for_stability)].copy()
stability_pivot = stability.pivot(index='variable', columns='model', values='coef_log_points_per_sd')
stability_range = stability.groupby('variable').agg(
    min_coef=('coef_log_points_per_sd','min'),
    max_coef=('coef_log_points_per_sd','max'),
    mean_coef=('coef_log_points_per_sd','mean'),
    sd_coef=('coef_log_points_per_sd','std'),
    sign_changes=('coef_log_points_per_sd', lambda s: int(len(set(np.sign(s.dropna()))) > 1)),
    min_p=('p_value','min'),
    max_p=('p_value','max'),
).reset_index()
stability_range['relative_range_abs'] = (stability_range['max_coef'] - stability_range['min_coef']).abs() / stability_range['mean_coef'].abs().replace(0, np.nan)
stability_range.to_csv(TABLE_DIR / 'rq3_main_model_core_coefficient_stability.csv', index=False)
stability_pivot.to_csv(TABLE_DIR / 'rq3_main_model_core_coefficients_wide.csv')
stability_range


In [ ]:

# Publication-style wide coefficient table for candidate comparison.
order = ['SES_raw', 'rural', 'education', 'female_share', 'age', 'Hhsize', 'log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD']
label_map = {
    'SES_raw': 'SES', 'rural': 'Rurality', 'education': 'Education', 'female_share': 'Female share',
    'age': 'Age', 'Hhsize': 'Household size', 'log_GDP_pc': 'log(GDP per capita)',
    'log_population': 'log(Population)', 'total_HDD': 'Heating degree days', 'total_CDD': 'Cooling degree days'
}
pub = []
for var in order:
    row = {'variable': label_map.get(var, var)}
    for model_name in candidate_specs:
        sub = coef_table[(coef_table['model'] == model_name) & (coef_table['variable'] == var)]
        if sub.empty:
            row[model_name] = ''
        else:
            s = sub.iloc[0]
            row[model_name] = f"{s['coef_log_points_per_sd']:.3f}{s['stars']} ({s['cluster_se']:.3f})"
    pub.append(row)
pub_table = pd.DataFrame(pub)
pub_table.to_csv(TABLE_DIR / 'rq3_main_model_candidate_publication_table.csv', index=False)
pub_table


In [ ]:

# Figures for model comparison.
plot_vars = ['SES_raw', 'rural', 'education', 'female_share', 'age', 'Hhsize']
plot_df = coef_table[coef_table['variable'].isin(plot_vars)].copy()
plot_df['variable_label'] = plot_df['variable'].map(label_map)
model_order = list(candidate_specs.keys())
plot_df['model'] = pd.Categorical(plot_df['model'], categories=model_order, ordered=True)

fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
y_positions = {v: i for i, v in enumerate(plot_vars)}
offsets = np.linspace(-0.24, 0.24, len(model_order))
colors = dict(zip(model_order, sns.color_palette('Set2', len(model_order))))
for mi, model_name in enumerate(model_order):
    sub = plot_df[plot_df['model'] == model_name]
    for _, row in sub.iterrows():
        y = y_positions[row['variable']] + offsets[mi]
        ax.errorbar(
            row['coef_log_points_per_sd'], y,
            xerr=[[row['coef_log_points_per_sd'] - row['ci_low']], [row['ci_high'] - row['coef_log_points_per_sd']]],
            fmt='o', color=colors[model_name], capsize=2.5, markersize=4.5, label=model_name if row['variable'] == plot_vars[0] else None
        )
ax.axvline(0, color='#333333', linewidth=1)
ax.set_yticks(range(len(plot_vars)))
ax.set_yticklabels([label_map[v] for v in plot_vars])
ax.invert_yaxis()
ax.set_xlabel('Coefficient on log(sufficiency ratio), per 1 SD predictor increase')
ax.set_title('RQ3 candidate main models: social predictor coefficients')
ax.legend(frameon=False, loc='lower right', fontsize=8)
fig.savefig(FIG_DIR / 'Figure_RQ3_candidate_main_models_social_coefficients.png', bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8), constrained_layout=True)
sns.barplot(data=summary_table, x='model', y='adj_r_squared', ax=axes[0], color='#4c78a8')
axes[0].set_title('Adjusted R²')
sns.barplot(data=summary_table, x='model', y='bic', ax=axes[1], color='#59a14f')
axes[1].set_title('BIC (lower is better)')
sns.barplot(data=summary_table, x='model', y='max_vif', ax=axes[2], color='#f28e2b')
axes[2].axhline(5, color='#777777', linestyle='--', linewidth=1)
axes[2].axhline(10, color='#333333', linestyle=':', linewidth=1)
axes[2].set_title('Max VIF')
for ax in axes:
    ax.tick_params(axis='x', rotation=35)
    ax.set_xlabel('')
fig.savefig(FIG_DIR / 'Figure_RQ3_candidate_main_models_fit_vif.png', bbox_inches='tight')
plt.close(fig)

summary_table


In [ ]:

# Manifest and concise summary for review before any interaction modelling.
manifest = {
    'data_file': str(DATA_PATH),
    'notebook': str(PROJECT / 'RQ3_main_model_selection.ipynb'),
    'results_dir': str(RESULTS_DIR),
    'process_dir': str(PROCESS_DIR),
    'common_sample_rows': int(len(sample)),
    'common_sample_countries': int(sample['iso_code'].nunique()),
    'candidate_models': candidate_specs,
    'controls': controls,
    'climate_controls': ['total_HDD', 'total_CDD'],
    'development_scale_controls': ['log_GDP_pc', 'log_population'],
    'standardized_coefficients': True,
    'fixed_effects': 'country fixed effects via C(iso_code)',
    'standard_errors': 'country-clustered standard errors',
    'outputs': {
        'summary': str(TABLE_DIR / 'rq3_main_model_candidate_summary.csv'),
        'coefficients': str(TABLE_DIR / 'rq3_main_model_candidate_coefficients_clustered.csv'),
        'vif': str(TABLE_DIR / 'rq3_main_model_candidate_vif.csv'),
        'spatial_morans_i': str(TABLE_DIR / 'rq3_main_model_candidate_spatial_morans_I.csv'),
        'publication_table': str(TABLE_DIR / 'rq3_main_model_candidate_publication_table.csv'),
    }
}
with open(PROCESS_DIR / 'rq3_main_model_selection_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
manifest
